<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-13/notebooks/ClimatePipeline/04_ClimateDailyConsolidator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateDailyConsolidator

Convierte el calendario auditado de precipitación en una sola fila por `estación + día`.

## Alcance

Esta primera versión procesa **exclusivamente precipitación**. La estructura del pipeline (calendario, manifiestos, particiones, trazabilidad y escritura segura) puede reutilizarse, pero las reglas de agregación, rangos válidos, outliers y sensores deben definirse y versionarse para cada variable. Por ejemplo, precipitación se acumula por día; humedad, temperatura, presión y viento no deben heredar automáticamente esa regla.

## Reglas del piloto

- Una ausencia permanece en `NaN`; nunca se convierte en cero.
- La cobertura candidata debe estar entre 90 % y 102 %.
- Los sensores con patrones instrumentales persistentes quedan en cuarentena.
- Los extremos aislados se conservan con bandera de revisión.
- Los sensores paralelos no se suman ni se promedian.
- Si varios sensores válidos difieren más de 0,1 mm, el valor aceptado queda en `NaN`.
- Si concuerdan, se prioriza `0240` y luego `0257`.

Los datos crudos, diarios preliminares y productos de auditoría son de solo lectura.

## 1. Preparar el repositorio

La celda clona o actualiza la rama para importar exactamente las reglas versionadas.

In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-13'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

## 2. Configuración protegida

La entrada predeterminada es la auditoría de los cuatro pilotos. Cambiar umbrales exige un nombre de consolidación distinto para conservar reproducibilidad.

In [ ]:
import json
import time

import pandas as pd

from ClimateProcessingUtils import (
    ahora_proyecto,
    detectar_commit,
    escribir_json_atomico,
    escribir_parquet_atomico,
    escribir_texto_atomico,
    formatear_duracion,
    slugificar,
)
from PrecipitationDailyAudit import AUDIT_VERSION
from PrecipitationDailyConsolidation import (
    CONSOLIDATION_VERSION,
    consolidar_precipitacion_diaria,
)

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

VARIABLE_NOMBRE = 'precipitacion'
DATASET_ID = 's54a-sgyg'
AUDITORIA_NOMBRE = 'piloto_2025_01_02'
CONSOLIDACION_NOMBRE = 'piloto_2025_01_02_v1'

COBERTURA_MINIMA_PCT = 90.0
COBERTURA_MAXIMA_PCT = 102.0
TOLERANCIA_SENSORES_MM = 0.1
PRIORIDAD_SENSORES = ['0240', '0257']
MINIMO_DIAS_CUARENTENA = 3

GUARDAR_RESULTADOS = True
SOBRESCRIBIR_CONSOLIDACION = False
EJECUTAR_CONSOLIDACION = False

PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'eco2026_processed'
)

if VARIABLE_NOMBRE != 'precipitacion':
    mensaje = (
        '⚠️ El notebook 04 solo tiene consolidación aprobada para precipitación. '
        'No use reglas de lluvia para otra variable; defina y pruebe su contrato propio.'
    )
    print(mensaje)
    raise NotImplementedError(mensaje)
AUDIT_INPUT_DIR = (
    PROCESSED_ROOT
    / 'auditorias_clima_diario'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'auditoria={slugificar(AUDITORIA_NOMBRE)}'
)
CONSOLIDATION_OUTPUT_DIR = (
    PROCESSED_ROOT
    / 'clima_diario_curado'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'consolidacion={slugificar(CONSOLIDACION_NOMBRE)}'
)

print({
    'regla_version': CONSOLIDATION_VERSION,
    'ejecutar': EJECUTAR_CONSOLIDACION,
    'guardar': GUARDAR_RESULTADOS,
    'sobrescribir': SOBRESCRIBIR_CONSOLIDACION,
    'entrada': str(AUDIT_INPUT_DIR),
    'salida': str(CONSOLIDATION_OUTPUT_DIR),
})

In [ ]:
def inspeccionar_entrada():
    manifest_path = AUDIT_INPUT_DIR / 'manifest.json'
    estado = 'NO_ENCONTRADA'
    contenido = {}
    if manifest_path.exists():
        contenido = json.loads(manifest_path.read_text(encoding='utf-8'))
        estado = contenido.get('estado', 'SIN_ESTADO')
    return pd.DataFrame([{
        'auditoria': AUDITORIA_NOMBRE,
        'estado': estado,
        'audit_version': contenido.get('audit_version'),
        'commit': contenido.get('commit'),
        'filas_calendario': contenido.get('metricas', {}).get('filas_calendario'),
        'filas_revision': contenido.get('metricas', {}).get('filas_revision'),
        'entrada': str(AUDIT_INPUT_DIR),
    }])


plan_df = inspeccionar_entrada()
display(Markdown('### Entrada de consolidación'))
display(plan_df)

## 3. Consolidación y trazabilidad

Solo se procesa una auditoría con manifiesto `COMPLETA`. La salida principal se particiona por departamento, año y mes.

In [ ]:
NOMBRES_ENTRADA = {
    'calendario': 'calendario_estacion_sensor.parquet',
    'sospechosos': 'valores_sospechosos.parquet',
    'manifest': 'manifest.json',
}
NOMBRES_SALIDA = {
    'diario': 'observaciones_estacion_dia.parquet',
    'candidatos': 'candidatos_sensor.parquet',
    'cuarentena': 'sensores_cuarentena.parquet',
    'resumen': 'resumen_calidad.parquet',
    'reporte': 'ConsolidacionDiaria_precipitacion_piloto_2025_01_02_v1.md',
    'manifest': 'manifest.json',
}


def cargar_auditoria_completa():
    manifest_path = AUDIT_INPUT_DIR / NOMBRES_ENTRADA['manifest']
    if not manifest_path.exists():
        raise FileNotFoundError(f'No existe el manifiesto de auditoría: {manifest_path}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('estado') != 'COMPLETA':
        raise RuntimeError(f'La auditoría no está completa: {AUDIT_INPUT_DIR}')
    if manifest.get('audit_version') != AUDIT_VERSION:
        raise RuntimeError(
            f"Versión de auditoría inesperada: {manifest.get('audit_version')} != {AUDIT_VERSION}."
        )
    calendario_path = AUDIT_INPUT_DIR / NOMBRES_ENTRADA['calendario']
    sospechosos_path = AUDIT_INPUT_DIR / NOMBRES_ENTRADA['sospechosos']
    calendario = pd.read_parquet(calendario_path)
    sospechosos = pd.read_parquet(sospechosos_path)
    filas_manifest = manifest.get('metricas', {}).get('filas_calendario')
    if filas_manifest is not None and len(calendario) != int(filas_manifest):
        raise RuntimeError(
            f'Filas de calendario ({len(calendario):,}) != manifiesto ({filas_manifest:,}).'
        )
    return calendario, sospechosos, manifest


def construir_resumen_calidad(diario):
    tabla = diario.copy()
    tabla['anio'] = tabla['fecha'].dt.year
    tabla['mes'] = tabla['fecha'].dt.month
    return (
        tabla.groupby(['departamento', 'anio', 'mes', 'calidad_dia'], as_index=False)
        .agg(
            filas=('fecha', 'size'),
            valores_aceptados=('precipitacion_diaria_mm', 'count'),
            dias_faltantes=('es_dia_faltante', 'sum'),
            filas_revision=('requiere_revision', 'sum'),
        )
    )


def tabla_markdown(tabla, limite=None):
    vista = tabla.head(limite) if limite is not None else tabla
    try:
        return vista.to_markdown(index=False)
    except ImportError:
        return '```text\n' + vista.to_string(index=False) + '\n```'


def construir_reporte(resultado, resumen, auditoria_manifest, inicio, fin, duracion):
    secciones = [
        '# Consolidación diaria de precipitación',
        '',
        f'- Regla: `{CONSOLIDATION_VERSION}`',
        f'- Auditoría de entrada: `{auditoria_manifest.get("commit")}`',
        f'- Commit consolidador: `{detectar_commit(REPO_DIR)}`',
        f'- Inicio: `{inicio.isoformat()}`',
        f'- Fin: `{fin.isoformat()}`',
        f'- Duración: `{formatear_duracion(duracion)}`',
        '',
        '## Parámetros',
        '',
        f'- Cobertura aceptable: `{COBERTURA_MINIMA_PCT} %` a `{COBERTURA_MAXIMA_PCT} %`.',
        f'- Tolerancia entre sensores: `{TOLERANCIA_SENSORES_MM} mm`.',
        f'- Prioridad de sensores: `{PRIORIDAD_SENSORES}`.',
        f'- Días mínimos para cuarentena: `{MINIMO_DIAS_CUARENTENA}`.',
        '',
        '> La ausencia y los desacuerdos permanecen en NaN. Los sensores nunca se suman ni se promedian.',
        '',
        '## Métricas',
        '',
        tabla_markdown(pd.DataFrame([resultado.metricas])),
        '',
        '## Calidad por partición',
        '',
        tabla_markdown(resumen),
        '',
        '## Sensores en cuarentena',
        '',
        tabla_markdown(resultado.sensores_cuarentena),
        '',
        '## Filas que requieren revisión',
        '',
        tabla_markdown(
            resultado.diario_estacion.loc[
                resultado.diario_estacion['requiere_revision'],
                [
                    'departamento', 'codigoestacion', 'fecha', 'sensor_seleccionado',
                    'precipitacion_diaria_mm', 'calidad_dia', 'motivos_revision',
                ],
            ],
            limite=50,
        ),
        '',
    ]
    return '\n'.join(secciones)


def guardar_figura_calidad(resumen, output_dir):
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print('Matplotlib no está disponible; se omite la figura.')
        return None
    pivot = resumen.pivot_table(
        index=['departamento', 'anio', 'mes'],
        columns='calidad_dia',
        values='filas',
        aggfunc='sum',
        fill_value=0,
    )
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab20c')
    ax.set_title('Calidad de la consolidación por partición')
    ax.set_ylabel('Filas estación-día')
    ax.set_xlabel('Departamento, año y mes')
    ax.legend(title='Calidad', bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    figures_dir = output_dir / 'figures'
    figures_dir.mkdir(parents=True, exist_ok=True)
    ruta = figures_dir / 'calidad_por_particion.png'
    fig.savefig(ruta, dpi=150)
    plt.show()
    plt.close(fig)
    return str(ruta)


def guardar_consolidacion(resultado, resumen, reporte, auditoria_manifest, inicio, fin, duracion):
    manifest_path = CONSOLIDATION_OUTPUT_DIR / NOMBRES_SALIDA['manifest']
    if manifest_path.exists() and not SOBRESCRIBIR_CONSOLIDACION:
        existente = json.loads(manifest_path.read_text(encoding='utf-8'))
        if existente.get('estado') == 'COMPLETA':
            print(f'La consolidación ya está completa; no se sobrescribe: {CONSOLIDATION_OUTPUT_DIR}')
            return existente
    if CONSOLIDATION_OUTPUT_DIR.exists() and any(CONSOLIDATION_OUTPUT_DIR.iterdir()) and not SOBRESCRIBIR_CONSOLIDACION:
        raise RuntimeError(f'Existe una consolidación incompleta: {CONSOLIDATION_OUTPUT_DIR}')

    escribir_json_atomico(
        {
            'estado': 'INICIADA',
            'regla_version': CONSOLIDATION_VERSION,
            'inicio': inicio.isoformat(),
            'commit': detectar_commit(REPO_DIR),
        },
        manifest_path,
        sobrescribir=SOBRESCRIBIR_CONSOLIDACION,
    )

    salidas_particiones = []
    diario = resultado.diario_estacion.copy()
    diario['anio'] = diario['fecha'].dt.year
    diario['mes'] = diario['fecha'].dt.month
    for (departamento, anio, mes), bloque in diario.groupby(['departamento', 'anio', 'mes'], sort=True):
        output_dir = (
            CONSOLIDATION_OUTPUT_DIR
            / f'departamento={departamento}'
            / f'anio={int(anio)}'
            / f'mes={int(mes):02d}'
        )
        ruta = output_dir / NOMBRES_SALIDA['diario']
        bloque = bloque.drop(columns=['anio', 'mes'])
        escribir_parquet_atomico(bloque, ruta, sobrescribir=SOBRESCRIBIR_CONSOLIDACION)
        salidas_particiones.append({
            'departamento': departamento,
            'anio': int(anio),
            'mes': int(mes),
            'ruta': str(ruta),
            'filas': len(bloque),
            'bytes': ruta.stat().st_size,
        })

    salidas_globales = {}
    for nombre, tabla in (
        ('candidatos', resultado.candidatos_sensor),
        ('cuarentena', resultado.sensores_cuarentena),
        ('resumen', resumen),
    ):
        ruta = CONSOLIDATION_OUTPUT_DIR / NOMBRES_SALIDA[nombre]
        escribir_parquet_atomico(tabla, ruta, sobrescribir=SOBRESCRIBIR_CONSOLIDACION)
        salidas_globales[nombre] = {'ruta': str(ruta), 'filas': len(tabla), 'bytes': ruta.stat().st_size}

    reporte_path = CONSOLIDATION_OUTPUT_DIR / NOMBRES_SALIDA['reporte']
    escribir_texto_atomico(reporte, reporte_path, sobrescribir=SOBRESCRIBIR_CONSOLIDACION)
    figura = guardar_figura_calidad(resumen, CONSOLIDATION_OUTPUT_DIR)
    manifest = {
        'estado': 'COMPLETA',
        'regla_version': CONSOLIDATION_VERSION,
        'commit': detectar_commit(REPO_DIR),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'auditoria_entrada': {
            'ruta': str(AUDIT_INPUT_DIR),
            'commit': auditoria_manifest.get('commit'),
            'audit_version': auditoria_manifest.get('audit_version'),
        },
        'parametros': {
            'cobertura_minima_pct': COBERTURA_MINIMA_PCT,
            'cobertura_maxima_pct': COBERTURA_MAXIMA_PCT,
            'tolerancia_sensores_mm': TOLERANCIA_SENSORES_MM,
            'prioridad_sensores': PRIORIDAD_SENSORES,
            'minimo_dias_cuarentena': MINIMO_DIAS_CUARENTENA,
        },
        'metricas': resultado.metricas,
        'particiones': salidas_particiones,
        'salidas_globales': salidas_globales,
        'reporte': str(reporte_path),
        'figura': figura,
    }
    escribir_json_atomico(manifest, manifest_path, sobrescribir=True)
    print(f'Consolidación guardada en: {CONSOLIDATION_OUTPUT_DIR}')
    return manifest

## 4. Ejecución protegida

Primero confirme que la auditoría de entrada aparezca `COMPLETA`. Luego cambie únicamente `EJECUTAR_CONSOLIDACION=True`.

In [ ]:
resultado_consolidacion = None

if not EJECUTAR_CONSOLIDACION:
    print('Consolidación desactivada. Revise la entrada y active EJECUTAR_CONSOLIDACION.')
else:
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    calendario, sospechosos, auditoria_manifest = cargar_auditoria_completa()
    print(f'Filas de calendario cargadas: {len(calendario):,}')
    resultado_consolidacion = consolidar_precipitacion_diaria(
        calendario,
        sospechosos,
        cobertura_minima_pct=COBERTURA_MINIMA_PCT,
        cobertura_maxima_pct=COBERTURA_MAXIMA_PCT,
        tolerancia_sensores_mm=TOLERANCIA_SENSORES_MM,
        prioridad_sensores=PRIORIDAD_SENSORES,
        minimo_dias_cuarentena=MINIMO_DIAS_CUARENTENA,
    )
    resumen_calidad = construir_resumen_calidad(resultado_consolidacion.diario_estacion)
    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    reporte = construir_reporte(
        resultado_consolidacion,
        resumen_calidad,
        auditoria_manifest,
        inicio,
        fin,
        duracion,
    )

    display(Markdown('## Métricas de consolidación'))
    display(pd.DataFrame([resultado_consolidacion.metricas]))
    display(Markdown('## Calidad por partición'))
    display(resumen_calidad)
    display(Markdown('## Sensores en cuarentena'))
    display(resultado_consolidacion.sensores_cuarentena)
    print(f'Duración: {formatear_duracion(duracion)}')

    if GUARDAR_RESULTADOS:
        guardar_consolidacion(
            resultado_consolidacion,
            resumen_calidad,
            reporte,
            auditoria_manifest,
            inicio,
            fin,
            duracion,
        )
    else:
        print('Resultados no guardados porque GUARDAR_RESULTADOS=False.')

## 5. Siguiente compuerta

La salida debe revisarse antes del agregado municipal. En particular:

1. Confirmar que las llaves `estación + fecha` sean únicas.
2. Verificar que febrero conserve los días ausentes como `NaN`.
3. Revisar la cuarentena y las siete discrepancias de Boyacá.
4. Confirmar que los extremos aislados válidos permanezcan disponibles y marcados.
5. Definir la geografía canónica de cada estación antes de asignarla a municipio.